In [1]:
import os  # 환경 변수 접근을 위한 os 모듈 임포트
from langchain_mcp_adapters.client import MultiServerMCPClient  # MCP 클라이언트 어댑터 임포트

# 환경 변수에서 GitHub Personal Access Token 가져오기
github_pat = os.getenv("GITHUB_PAT")

# GitHub 서버 구성으로 MCP 클라이언트 생성
mcp_client = MultiServerMCPClient({
    #Streamable HTTP transport 옵션
    "github": {
      "url": "https://api.githubcopilot.com/mcp/",  # GitHub Copilot MCP API 엔드포인트
      "headers": {
        "Authorization": f"Bearer {github_pat}"  # Bearer 토큰 인증
      },
      "transport": "streamable_http"  # 새로운 표준 transport 방식
    }
})

In [3]:
tool_list = await mcp_client.get_tools()

In [4]:
tool_list

[StructuredTool(name='add_comment_to_pending_review', description="Add review comment to the requester's latest pending pull request review. A pending review needs to already exist to call this (check with the user if not sure).", args_schema={'type': 'object', 'properties': {'body': {'type': 'string', 'description': 'The text of the review comment'}, 'line': {'type': 'number', 'description': 'The line of the blob in the pull request diff that the comment applies to. For multi-line comments, the last line of the range'}, 'owner': {'type': 'string', 'description': 'Repository owner'}, 'path': {'type': 'string', 'description': 'The relative path to the file that necessitates a comment'}, 'pullNumber': {'type': 'number', 'description': 'Pull request number'}, 'repo': {'type': 'string', 'description': 'Repository name'}, 'side': {'type': 'string', 'description': 'The side of the diff to comment on. LEFT indicates the previous state, RIGHT indicates the new state', 'enum': ['LEFT', 'RIGHT']

In [5]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI


load_dotenv()

llm = ChatOpenAI(model='gpt-4o')

In [6]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tool_list,
    system_prompt=("Use the tools provided to you to answer the user's question")
)

In [7]:
async def process_stream(stream_generator):
    results = []
    try:
        async for chunk in stream_generator:

            key = list(chunk.keys())[0]
            
            if key == 'agent':
                # Agent 메시지의 내용을 가져옴. 메세지가 비어있는 경우 어떤 도구를 어떻게 호출할지 정보를 가져옴
                content = chunk['agent']['messages'][0].content if chunk['agent']['messages'][0].content != '' else chunk['agent']['messages'][0].additional_kwargs
                print(f"'agent': '{content}'")
            
            elif key == 'tools':
                # 도구 메시지의 내용을 가져옴
                for tool_msg in chunk['tools']['messages']:
                    print(f"'tools': '{tool_msg.content}'")
            
            results.append(chunk)
        return results
    except Exception as e:
        print(f"Error processing stream: {e}")
        return results

In [9]:
from langchain_core.messages import HumanMessage

query = """깃헙의 Pull Request를 확인하고 코드 리뷰를 작성해주세요. 
코드리뷰를 PR에 comment로 남겨주세요. 단, 특정 라인 댓글이 아닌 add_issue_comment로 전체 리뷰 댓글을 달아주세요.

PR URL:https://github.com/hnjee/tax-chatbot/pull/1

"""

stream_generator = agent.astream({'messages': [HumanMessage(content=query)]})


all_chunks = await process_stream(stream_generator)


if all_chunks:
    final_result = all_chunks[-1]
    print(final_result)

'tools': '[{'type': 'text', 'text': 'diff --git a/main.py b/main.py\nindex d012c40..7b15b74 100644\n--- a/main.py\n+++ b/main.py\n@@ -28,6 +28,7 @@ def main():\n                 ai_message = st.write_stream(ai_response)\n                 st.session_state.message_list.append({"role": "ai", "content": ai_message})\n \n-\n+    #PR을 위한 주석입니다\n+    \n if __name__ == "__main__":\n     main()\n', 'id': 'lc_b9e00107-9a0a-4992-9e4a-c4f84f288fb0'}]'
'tools': '[{'type': 'text', 'text': '{"id":"4087606310","url":"https://github.com/hnjee/tax-chatbot/pull/1#issuecomment-4087606310"}', 'id': 'lc_7ccf7c5d-5a9b-4289-b26d-0812de3665ac'}]'
{'model': {'messages': [AIMessage(content='코드 리뷰를 완료하고 PR에 코멘트를 남겼습니다. 코멘트는 [이 링크](https://github.com/hnjee/tax-chatbot/pull/1#issuecomment-4087606310)에서 확인하실 수 있습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 6419, 'total_tokens': 6474, 'completion_tokens_details': {'accepted_prediction_tokens':